# LogoGen: Generate Logos with Diffusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivtx/ai-miden/blob/main/logogen/logogen_colab.ipynb)
[![GitHub](https://img.shields.io/badge/github-repo-blue)](https://github.com/srivtx/ai-miden/tree/main/logogen)

A small diffusion model (DDPM + U-Net) that generates logo-like images.

**One-click workflow:** `Runtime` -> `Run all`. Downloads ~300 logos from public image search, trains a small U-Net on T4 GPU (30-60 min), and generates 40 new logos.

No API keys. No manual curation. No authentication.

## Step 1: Verify GPU

`Runtime` -> `Change runtime type` -> T4 GPU -> Save.

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. CPU training is 50-100x slower.")
    print("Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Get the code

In [ ]:
!git clone https://github.com/srivtx/ai-miden.git
%cd ai-miden/logogen
!ls -la

## Step 3: Smoke test (verify the pipeline works)

Generates 20 synthetic geometric images, trains 200 steps, generates 4 test logos. Runs in 30 seconds. If this fails, something is wrong with the environment.

In [ ]:
!python logogen.py smoke --out-dir /content/logogen_test --max-steps 200

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("/content/logogen_test/smoke_test.png"):
    print("Smoke test generated logos:")
    display(Image("/content/logogen_test/smoke_test.png"))
else:
    print("Smoke test failed. Check the cell above.")

## Step 4: Download logos

Generates 300 synthetic geometric logos (colored shapes, letters, line art). Always works. Optionally uses Pexels API for real logo images (set `PEXELS_API_KEY` env var, free at pexels.com/api).

In [ ]:
!python download.py --out-dir /content/logos --n 300

In [ ]:
# Preview a few downloaded logos
import os, random
from PIL import Image
from IPython.display import display

logos = [f for f in os.listdir("/content/logos") if f.endswith(".png")]
if logos:
    print(f"Downloaded {len(logos)} logos. Preview:")
    for f in random.sample(logos, min(6, len(logos))):
        img = Image.open(os.path.join("/content/logos", f))
        display(img.resize((128, 128)))
else:
    print("No logos found. Check the download cell output.")

## Step 5: Train

~30-60 min on T4. Loss drops from ~0.5 to ~0.02. Sample grids are saved every few epochs to `/content/logogen/`.

In [ ]:
!python logogen.py train \
    --image-dir /content/logos \
    --out-dir /content/logogen \
    --image-size 128 \
    --epochs 50 \
    --max-steps 50000 \
    --batch-size 16 \
    --lr 2e-4

## Step 6: Generate new logos

Generates 40 new logos (5 batches of 8) and saves to `/content/generated/`.

In [ ]:
!python logogen.py sample \
    --model /content/logogen/final.pt \
    --out-dir /content/generated \
    --n 8 \
    --n-batches 5 \
    --image-size 128

In [ ]:
# Show generated logos
import os
from PIL import Image
from IPython.display import display

for f in sorted(os.listdir("/content/generated")):
    if f.endswith(".png"):
        img = Image.open(os.path.join("/content/generated", f))
        print(f"  {f}")
        display(img)

## What's next

**Better quality:** collect your own logos manually (Pinterest/Behance, 300+), re-upload to `/content/logos/`, re-run training.

**Larger images:** change `--image-size 256` in both train and sample (reduces batch-size to 4).

**Faster test:** reduce `--epochs 10` and `--max-steps 5000`.

**Use a specific style:** pick a single search query like `musical party event poster design` and use `--n 500`.